In [65]:
import random
import operator
import pandas as pd
from tabulate import tabulate 

In [66]:
"""
Класс, реализующий арифметические операции над целыми числами, представленными
символическими последовательностями 
"""

class Path:
    def __init__(self, symbols=None, number=None):
        valid_chars = {'S', 'P'}

        if symbols is not None and number is not None:
            # В случае задания сразу двух непустых аргументов вызываем исключение
            raise ValueError('Uncertainty in the way the path in definition')
        elif symbols is not None:
            if all(char in valid_chars for char in symbols):
                self.symbols = symbols if symbols else []
            else:
                raise ValueError(f'Only {valid_chars} in {symbols} allowed')
        elif number is not None:
            if isinstance(number, int):
                self.symbols = self.from_int(number)
            else:
                raise ValueError(f'Only int number allowed')
        else:
            self.symbols = []
    
    def from_int(self, n):
        """Конвертация int -> Path"""
        return ['S' if n > 0 else 'P' for _ in range(n)]

    def normalize(self):
        """Удаляет соседние SP/PS пары."""
        stack = []
        for s in self.symbols:
            if stack and ((stack[-1] == 'S' and s == 'P') or (stack[-1] == 'P' and s == 'S')):
                stack.pop()
            else:
                stack.append(s)
        return Path(stack)

    def simplify(self, path):
        """
        Упрощает путь, удаляя все циклы. 
        Возвращает либо пустой путь, либо путь из одинаковых символов
        """

        result_list = list(path)  # Convert the string to a list for easy removal
        index = 0
        while index < len(result_list) - 1:
            if result_list[index] != result_list[index + 1]:
                result_list.pop(index + 1)  # Remove the second character
                result_list.pop(index)      # Remove the first character
                index = 0  # Go back to the beginning to handle possible new pairs
            else:
                index += 1  # Move to the next character

        return Path("".join(result_list))  # Convert the list back to a string

    def __add__(self, other):
        """Сложение: конкатенация """
        return Path(self.symbols + other.symbols)

    def __sub__(self, b):
        """Вычитание"""
        neg_b = b.__neg__()
        return Path(self.symbols + neg_b.symbols)

    def __mul__(self, b):
        """Умножение"""
        return Path(self.mult_paths(self.symbols, b.symbols))
    
    def __neg__(self):
        """Инверсия: обратный порядок и замена S ↔ P."""
        return Path("".join([ 'P' if s == 'S' else 'S' for s in reversed(self.symbols)]))

    def inv(self, path):
        """Инвертирует путь (заменяет 'a' на 'b' и наоборот)."""
        return "".join([('P' if c == 'S' else 'S') for c in path])

    def reverse(self, path):
        """Разворачивает путь."""
        return path[::-1]

    def simplify_path(self, path):
        """Упрощает путь, удаляя соседние обратные элементы."""
        result = []
        for char in path:
            if result and (
                (result[-1] == 'S' and char == 'P') or
                (result[-1] == 'P' and char == 'S')
            ):
                result.pop()
            else:
                result.append(char)
        return "".join(result)

    def path_sum(self, path):
        """Вычисляет сумму пути (a = +1, b = -1)."""
        return sum([1 if c == 'S' else -1 for c in path])
    

    def abs(self, path):
        return "".join(self.reverse(self.inv(path))) if self.path_sum(path) < 0 else "".join(path)
    
    def mult_paths(self, a, b):
        dq = ''
        for _ in range(abs(self.path_sum(a))):
            dq += b

        if self.path_sum(a) < 0:
            dq = self.inv(dq)
        
        return dq  

    def divide(self, divisor):
        """Делит путь self.symbols на путь divisor с остатком"""
    
        # Случай, когда оба числа имеют одинаковый знак (оба положительные или оба отрицательные)
        dividend_num = self.path_sum(self.symbols)
        divisor_num = self.path_sum(divisor.symbols)

        if divisor_num == 0:
            # Вычисляем энтропию делителя
            divisor_entropy = self.entropy(divisor.symbols)
            
            # Определяем частное на основе энтропии
            # Чем выше энтропия, тем больше "информации" в делителе
            quotient_value = int(divisor_entropy * 10) + 1  # +1, чтобы избежать нулевого значения
            
            # Учитываем знак делимого
            quotient = 'S' * quotient_value if self.path_sum(self.symbols) >= 0 else 'P' * quotient_value
            
            # Вычисляем остаток
            remainder = self.symbols
            delta = self.reverse(self.inv(divisor.symbols))
            for _ in range(quotient_value):
                remainder = remainder + delta
            
            return Path(quotient), Path(remainder)
        
        if (dividend_num >= 0 and divisor_num > 0) or (dividend_num <= 0 and divisor_num < 0):
            abs_dividend = self.abs(self.symbols)
            abs_divisor = self.abs(divisor.symbols)
            
            quotient = ''
            remainder = abs_dividend
            delta = self.reverse(self.inv(abs_divisor))
            
            while self.path_sum(remainder) >= abs(divisor_num):
                remainder = self.simplify_path(remainder + delta)
                quotient += 'S'
            
            # Восстанавливаем знаки
            if dividend_num < 0 and divisor_num < 0:
                remainder = self.inv(remainder)
            
            return Path(quotient), Path(remainder)
        
        # Случай, когда числа имеют разные знаки
        else:
            abs_divisor = self.abs(divisor.symbols)
            
            # Находим наибольшее кратное делителю, которое не превышает делимое
            quotient = ''
            
            if dividend_num > 0 and divisor_num < 0:
                # Делимое положительное, делитель отрицательный
                while dividend_num - (self.path_sum(quotient) + 1) * divisor_num > abs(self.path_sum(divisor.symbols)):
                    quotient += 'P'
                
                dq = self.mult_paths(quotient, divisor.symbols)
                remainder = self.symbols + self.reverse(self.inv(dq))  
                
            else:  # dividend < 0 and divisor > 0
                # Делимое отрицательное, делитель положительный
                while dividend_num - (self.path_sum(quotient) + 1) * divisor_num < -abs(self.path_sum(divisor.symbols)):
                    quotient += 'P'
                
                dq = self.mult_paths(quotient, divisor.symbols)
                remainder = self.symbols + self.reverse(self.inv(dq))  
            
            return Path(quotient), Path(remainder)

    def entropy(self, path):
        """Вычисляет энтропию последовательности как меру ее сложности"""
        if not path:
            return 0
        
        # Подсчитываем количество переключений между S и P
        switches = 0
        for i in range(1, len(path)):
            if path[i] != path[i-1]:
                switches += 1
        
        # Вычисляем энтропию как отношение числа переключений к длине
        return switches / len(path)

    def __floordiv__(self, divisor):
        """Переопределение операции //"""
        return self.divide(divisor)[0]

    def __mod__(self, divisor):
        """Переопределение операции %"""
        return self.divide(divisor)[1]
    
    def __repr__(self):
        return ''.join(self.symbols)
    
    def __int__(self):
        simple_path = self.simplify(self.symbols)
        n = len(simple_path.symbols)
        
        if n == 0:
            return 0
        elif simple_path.symbols[0] == 'S':
            return n
        else:
            return -n        
        

In [67]:
def is_loop(path):
    """Возвращает True, если путь является петлей"""
    return int(path) == 0

def generate_random_paths(length):
    """
    Генерация случайных путей длины length
    """
    generated_string = ''.join(random.choice(['S', 'P']) for _ in range(length))
    return Path(generated_string)

def generate_random_loops(length):
    """
    Генерация случайных петель длины length
    """
    if length % 2 != 0:
        generated_string = ''  # String length must be even
    else:
        half_length = length // 2
        character_list = ['S'] * half_length + ['P'] * half_length
        random.shuffle(character_list)
        generated_string = "".join(character_list)
    return Path(generated_string)

In [68]:
def random_numbers_divide(a_rnd_generator, b_rnd_generator, a_path_len, b_path_len, num):
    results = []

    for _ in range(num):
        a_p = a_rnd_generator(a_path_len)
        b_p = b_rnd_generator(b_path_len)
        a_div_b = a_p // b_p
        a_mod_b = a_p % b_p
        ex = {
            'a': int(a_p),
            'a (путь)': a_p,
            'b': int(b_p),
            'b (путь)': b_p,
            'a // b': int(a_div_b),
            'a mod b': int(a_mod_b),
            'a // b (путь)': a_div_b,
            'a mod b (путь)': a_mod_b
        }
        results.append(ex)   

    return pd.DataFrame(results) 

In [69]:
def random_numbers_operation(operation, a_rnd_generator, b_rnd_generator, a_path_len, b_path_len, num):
    results = []

    for _ in range(num):
        a_p = a_rnd_generator(a_path_len)
        b_p = b_rnd_generator(b_path_len)
        a_plus_b = operation(a_p, b_p)
        ex = {
            'a': int(a_p),
            'a (путь)': a_p,
            'b': int(b_p),
            'b (путь)': b_p,
            'a + b': int(a_plus_b),
            'a + b (путь)': a_plus_b
        }
        results.append(ex)   

    return pd.DataFrame(results) 

In [70]:
# Сложение 10 разных случайных пар ненулевых чисел
results = random_numbers_operation(operator.add, generate_random_paths, generate_random_paths, 13, 9, 10)
print(tabulate(results, showindex=False, headers=results.columns))

  a  a (путь)         b  b (путь)      a + b  a + b (путь)
---  -------------  ---  ----------  -------  ----------------------
 -1  SPPSSPPPSSPSP   -1  SPSPSSPPP        -2  SPPSSPPPSSPSPSPSPSSPPP
  3  PPSSSSSSPSPPS   -1  PSPPSSPSP         2  PPSSSSSSPSPPSPSPPSSPSP
  1  PPSSSPSSPPSPS   -1  PSSSPPPPS         0  PPSSSPSSPPSPSPSSSPPPPS
 -3  PPPSPPPPSPSSS    1  PPSPPSSSS        -2  PPPSPPPPSPSSSPPSPPSSSS
  5  SSSPPPSSSSSPS    1  PSSPSSPSP         6  SSSPPPSSSSSPSPSSPSSPSP
 -5  PPSSSPPPPPPPS   -1  PSSPSPSPP        -6  PPSSSPPPPPPPSPSSPSPSPP
  1  SPPSSSSPPPSSP    1  PSSSPPSSP         2  SPPSSSSPPPSSPPSSSPPSSP
 -3  SPPPPPPSSPSPS    3  SSSPPSSSP         0  SPPPPPPSSPSPSSSSPPSSSP
 -1  SPPSSPPSPSPSP    3  PPPSSSSSS         2  SPPSSPPSPSPSPPPPSSSSSS
 -3  SPSSPPPPPSSPP    1  PSSPPSPSS        -2  SPSSPPPPPSSPPPSSPPSPSS


In [71]:
# Вычитание 10 разных случайных пар ненулевых чисел
results = random_numbers_operation(operator.sub, generate_random_paths, generate_random_paths, 13, 9, 10)
print(tabulate(results, showindex=False, headers=results.columns))

  a  a (путь)         b  b (путь)      a + b  a + b (путь)
---  -------------  ---  ----------  -------  ----------------------
  1  SPPPSSSSSPSPP    5  SSSSPSPSS        -4  SPPPSSSSSPSPPPPSPSPPPP
 -1  PPSSSSPSPPSPP   -5  PPSSPPPPP         4  PPSSSSPSPPSPPSSSSSPPSS
 -1  PPPSPSSSPPSSP    3  SSSPSPPSS        -4  PPPSPSSSPPSSPPPSSPSPPP
  7  PPSSSSSSSSSPS    3  PSPSSSPSS         4  PPSSSSSSSSSPSPPSPPPSPS
  3  SPPSPSPSPSSSS   -1  PPPSSPPSS         4  SPPSPSPSPSSSSPPSSPPSSS
 -5  PPSPPSPPPSPPS    3  PPSSSSSPS        -8  PPSPPSPPPSPPSPSPPPPPSS
 -1  PSPSPSSPPPSSP   -1  SSPSPPSPP         0  PSPSPSSPPPSSPSSPSSPSPP
 -5  SPPPSPPPSSPPP    3  SSPSPPSSS        -8  SPPPSPPPSSPPPPPPSSPSPP
  1  SPPSPPSPSPSSS   -1  PSPSPPPSS         2  SPPSPPSPSPSSSPPSSSPSPS
 -1  PPPSPPSSPPSSS   -5  PSPPPSPPP         4  PPPSPPSSPPSSSSSSPSSSPS


In [72]:
# Умножение 10 разных случайных пар ненулевых чисел
results = random_numbers_operation(operator.mul, generate_random_paths, generate_random_paths, 13, 9, 10)
print(tabulate(results, showindex=False, headers=results.columns))

  a  a (путь)         b  b (путь)      a + b  a + b (путь)
---  -------------  ---  ----------  -------  ---------------------------------------------------------------
  1  SPPPSSSSPPSPS    7  SSSSSPSSS         7  SSSSSPSSS
  1  SPSPPSPSPSSSP    3  PPPSSSSSS         3  PPPSSSSSS
  3  SSPPPSPSPSSSS    1  PPSPPSSSS         3  PPSPPSSSSPPSPPSSSSPPSPPSSSS
  7  SSSSSSSSPPSPS    1  PSSSSPPPS         7  PSSSSPPPSPSSSSPPPSPSSSSPPPSPSSSSPPPSPSSSSPPPSPSSSSPPPSPSSSSPPPS
  1  SSSSSPSPPPSPP    3  SPSSPPSSS         3  SPSSPPSSS
 -1  SSPSSPPPPSPPS   -1  PPSSPPSPS         1  SSPPSSPSP
 -1  PPSSPPSPSSSPP    1  PPSSPPSSS        -1  SSPPSSPPP
 -5  SSPPPPPPPSPPS   -1  SPSSPSPPP         5  PSPPSPSSSPSPPSPSSSPSPPSPSSSPSPPSPSSSPSPPSPSSS
  3  PSPSSPPPSSSSS    5  SSPSSPSSS        15  SSPSSPSSSSSPSSPSSSSSPSSPSSS
  1  PPSPSSPSPPSSS   -1  SSPPPPPSS        -1  SSPPPPPSS


In [10]:
# Деление 10 разных случайных пар ненулевых чисел
results = random_numbers_divide(generate_random_paths, generate_random_paths, 11, 15, 10)
print(tabulate(results, showindex=False, headers=results.columns))

  a  a (путь)       b  b (путь)           a // b    a mod b  a // b (путь)    a mod b (путь)
---  -----------  ---  ---------------  --------  ---------  ---------------  --------------------------
  5  SSSSPPSSPSS   -5  SPSSPPPPPSPPPPS        -1          0  P                SSSSPPSSPSSSPPPPSPPPPPSSPS
  1  PPPSSSPSPSS   -1  SSSPSSPSPPPPPSP        -1          0  P                PPPSSSPSPSSPSPPPPPSPSSPSSS
  1  SSSSPSSPPPP   -7  PSPPPPSSPPPPPPS        -1         -6  P                SSSSPSSPPPPSPPPPPPSSPPPPSP
  1  SSSSPSPPPPS    5  SSSPSSPPPSSSSSP         0          1                   SSSSPSPPPPS
 -5  PPSPPPPSSPP   -5  SPSPSPPPPPPPSPS         1          0  S
  3  SSPPSSSPSSP    7  SPSSPSPSPSSSSSS         0          3                   SSPPSSSPSSP
 -1  PSPSPSPPPSS   -5  PSSPSPSPPPSPPPP         0         -1                   SSPPPSPSPSP
 -1  SPSSSPPPSPP   -3  SSPSSPPPPPPSPSP         0         -1                   PPSPPPSSSPS
  1  PSPSPSPSSSP   -3  PSPSPSPPSSPPSPP        -1         -2  P  

In [11]:
# Деление 10 разных случайных ненулевых чисел на 10 случайных нулей с разной структурой
results = random_numbers_divide(generate_random_paths, generate_random_loops, 9, 14, 10)
print(tabulate(results, showindex=False, headers=results.columns))

  a  a (путь)      b  b (путь)          a // b    a mod b  a // b (путь)    a mod b (путь)
---  ----------  ---  --------------  --------  ---------  ---------------  -------------------------------------------------------------------------------------------------------------------------
 -3  SSSPPPPPP     0  SSPPPSPSPPSPSS        -6         -3  PPPPPP           SSSPPPPPPPPSPSSPSPSSSPPPPSPSSPSPSSSPPPPSPSSPSPSSSPPPPSPSSPSPSSSPPPPSPSSPSPSSSPPPPSPSSPSPSSSPP
  1  SPSSPSPPS     0  PPSSSSSSPPPPPS         3          1  SSS              SPSSPSPPSPSSSSSPPPPPPSSPSSSSSPPPPPPSSPSSSSSPPPPPPSS
 -1  PSSPSSPPP     0  PSSPPPPSSSSSPP        -3         -1  PPP              PSSPSSPPPSSPPPPPSSSSPPSSSPPPPPSSSSPPSSSPPPPPSSSSPPS
 -3  PPSPPPSSP     0  PPSPPSPPSSPSSS        -6         -3  PPPPPP           PPSPPPSSPPPPSPPSSPSSPSSPPPSPPSSPSSPSSPPPSPPSSPSSPSSPPPSPPSSPSSPSSPPPSPPSSPSSPSSPPPSPPSSPSSPSS
 -7  PPPPPPSPP     0  PPSSSPPPPSPSSS        -4         -7  PPPP             PPPPPPSPPPPPSPSSSSPPPSSPPPSPSSSSPPPSSPP

In [12]:
# Деление 10 разных случайных пар нулей с разной структурой
results = random_numbers_divide(generate_random_loops, generate_random_loops, 6, 14, 10)
print(tabulate(results, showindex=False, headers=results.columns))

  a  a (путь)      b  b (путь)          a // b    a mod b  a // b (путь)    a mod b (путь)
---  ----------  ---  --------------  --------  ---------  ---------------  --------------------------------------------------------------------------------------------------------
  0  PPSSPS        0  PPSPSSPSSPPSSP         6          0  SSSSSS           PPSSPSSPPSSPPSPPSPSSSPPSSPPSPPSPSSSPPSSPPSPPSPSSSPPSSPPSPPSPSSSPPSSPPSPPSPSSSPPSSPPSPPSPSS
  0  SSPPSP        0  PSPSSSPPSSSPPP         5          0  SSSSS            SSPPSPSSSPPPSSPPPSPSSSSPPPSSPPPSPSSSSPPPSSPPPSPSSSSPPPSSPPPSPSSSSPPPSSPPPSPS
  0  PPSSSP        0  SPSPSPSPPPPSSS         6          0  SSSSSS           PPSSSPPPPSSSSPSPSPSPPPPSSSSPSPSPSPPPPSSSSPSPSPSPPPPSSSSPSPSPSPPPPSSSSPSPSPSPPPPSSSSPSPSPSP
  0  SPPPSS        0  SPPPSSSSSPPSPP         4          0  SSSS             SPPPSSSSPSSPPPPPSSSPSSPSSPPPPPSSSPSSPSSPPPPPSSSPSSPSSPPPPPSSSP
  0  PSPSPS        0  PPSSSPPPSPSPSS         6          0  SSSSSS           PSPSPSPPSPSPSSSPPPSSPPSPSP